# Raynaud's CREST Pipeline
**Transcriptome-Guided Drug Repurposing for Raynaud's Phenomenon in Systemic Sclerosis**

Glen Ritschel | Ritschel Research | 2026

GitHub: https://github.com/glenritschel/raynaud-crest

## Overview
This notebook orchestrates the full scRNA-seq drug repurposing pipeline for Raynaud's phenomenon in SSc:
1. Load GSE138669 (Tabib 2021, 22 SSc skin biopsies)
2. QC + marker-based vascular cell isolation (ECs + pericytes)
3. scVI embedding + multi-resolution Leiden clustering (GPU)
4. EC subtype annotation (arterial, capillary, venous, lymphatic, pericyte)
5. Raynaud's signature scoring (vasospasm, EC injury, angiogenesis, oxidative stress, pericyte dysfunction)
6. Wilcoxon DE + LINCS L1000 reversal scoring
7. Novelty assessment + priority ranking

**Runtime**: ~45-90 min on T4 GPU (scVI training dominates)  
**Key improvement over calcinosis pipeline**: vascular cell isolation before scVI; multi-resolution Leiden; EC subtype annotation

---
**Before running**: Runtime > Change runtime type > T4 GPU

## Cell 1: Clone repo and install dependencies

In [ ]:
# Clone the pipeline repo
import os

REPO_URL = "https://github.com/glenritschel/raynaud-crest"
REPO_DIR = "raynaud-crest"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}/notebooks
print(f"Working directory: {os.getcwd()}")
print("Files:", os.listdir("."))

## Cell 2: Download raw data from NCBI GEO

In [ ]:
# Download GSE138669 .h5 files directly from NCBI GEO
# This bypasses Google Drive entirely
import os, urllib.request, glob

REPO_H5_DIR = "raynaud-crest/data/raw/GSE138669"
os.makedirs(REPO_H5_DIR, exist_ok=True)

# The 22 GSE138669 samples with their GEO accession and subject IDs
SAMPLES = [
    ("GSM4115868", "SC1"),  ("GSM4115869", "SC2"),
    ("GSM4115870", "SC4"),  ("GSM4115871", "SC5"),
    ("GSM4115872", "SC18"), ("GSM4115873", "SC19"),
    ("GSM4115874", "SC32"), ("GSM4115875", "SC33"),
    ("GSM4115876", "SC34"), ("GSM4115877", "SC49"),
    ("GSM4115878", "SC50"), ("GSM4115879", "SC60"),
    ("GSM4115880", "SC68"), ("GSM4115881", "SC69"),
    ("GSM4115882", "SC70"), ("GSM4115883", "SC86"),
    ("GSM4115884", "SC119"),("GSM4115885", "SC124"),
    ("GSM4115886", "SC125"),("GSM4115887", "SC185"),
    ("GSM4115888", "SC188"),("GSM4115889", "SC189"),
]

# Dev mode: set to a small number to test with fewer files
# Match whatever N_DEV_SAMPLES is set to in the config cell
# Set to None to download all 22
N_DOWNLOAD = N_DEV_SAMPLES  # uses value from config cell above
if N_DOWNLOAD:
    samples_to_get = SAMPLES[:N_DOWNLOAD]
else:
    samples_to_get = SAMPLES

print(f"Downloading {len(samples_to_get)} samples from NCBI GEO...")
base = "https://ftp.ncbi.nlm.nih.gov/geo/samples"

for gsm, subj in samples_to_get:
    fname = f"{gsm}_{subj}raw_feature_bc_matrix.h5"
    dest = os.path.join(REPO_H5_DIR, fname)
    if os.path.exists(dest):
        print(f"  Already present: {fname}")
        continue
    # GEO FTP path structure: /geo/samples/GSM4115nnn/GSM4115868/suppl/
    prefix = gsm[:7] + "nnn"
    url = f"{base}/{prefix}/{gsm}/suppl/{fname}"
    print(f"  Downloading {fname}...", end=" ", flush=True)
    try:
        urllib.request.urlretrieve(url, dest)
        size_mb = os.path.getsize(dest) / 1e6
        print(f"{size_mb:.0f} MB")
    except Exception as e:
        print(f"FAILED: {e}")

downloaded = glob.glob(os.path.join(REPO_H5_DIR, "*.h5"))
print(f"
Done. {len(downloaded)} .h5 files ready in {REPO_H5_DIR}")


In [ ]:
# Install dependencies (run once per session)
%run ../src/00_install.py

In [ ]:
# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. scVI will run on CPU (much slower).")
    print("Go to Runtime > Change runtime type > T4 GPU")

## Cell 2: Configuration
Edit these settings before running the full pipeline.

In [ ]:
# ── Pipeline configuration ────────────────────────────────────────────────────

# Set to an integer (e.g. 2) for a fast dev run on 2 samples
# Set to None for the full 22-sample run
N_DEV_SAMPLES = None  # None = full run

# Patch the config into 01_load_qc.py at runtime
import re

def patch_config(script_path, var_name, new_value):
    with open(script_path) as f:
        content = f.read()
    # Replace the variable assignment
    pattern = rf'^({re.escape(var_name)}\s*=\s*).*$'
    replacement = rf'\g<1>{repr(new_value)}'
    new_content = re.sub(pattern, replacement, content, flags=re.MULTILINE)
    with open(script_path, 'w') as f:
        f.write(new_content)
    print(f"  Patched {script_path}: {var_name} = {new_value!r}")

patch_config("../src/01_load_qc.py", "N_DEV_SAMPLES", N_DEV_SAMPLES)
print("Configuration set.")

## Cell 3: Script 01 — Load, QC, Vascular Isolation

In [ ]:
%run ../src/01_load_qc.py

In [ ]:
# QC checkpoint
import scanpy as sc
import pandas as pd

adata_vasc = sc.read_h5ad("data/adata_vascular.h5ad")
print(f"Vascular subset: {adata_vasc.n_obs} cells × {adata_vasc.n_vars} genes")
print(f"Samples: {adata_vasc.obs['sample'].nunique()}")
print(f"\nVascular subtype breakdown:")
print(adata_vasc.obs['vascular_subtype_prelim'].value_counts())

# Flag if too few vascular cells
if adata_vasc.n_obs < 200:
    print("\nWARNING: Very few vascular cells. Consider:")
    print("  1. Lowering VASCULAR_EXPR_THRESHOLD in 01_load_qc.py (currently 0.5)")
    print("  2. Lowering VASCULAR_MIN_MARKERS to 1 (already at minimum)")
    print("  3. Adding more permissive EC markers to EC_MARKERS list")
else:
    print("\nCell count OK for scVI training.")

## Cell 4: Script 02 — scVI Embedding + Multi-Resolution Leiden
**This is the GPU step. Expect ~30-60 min on T4.**

In [ ]:
%run ../src/02_scvi_embed.py

In [ ]:
# scVI checkpoint
import scanpy as sc
import pandas as pd

adata_scvi = sc.read_h5ad("data/adata_scvi.h5ad")
res_metrics = pd.read_csv("data/resolution_metrics.csv")

recommended_res = adata_scvi.uns.get('recommended_leiden_resolution', 'unknown')
n_clusters = adata_scvi.obs['leiden'].nunique()

print(f"scVI latent shape: {adata_scvi.obsm['X_scVI'].shape}")
print(f"Recommended resolution: {recommended_res}")
print(f"Final clusters: {n_clusters}")
print(f"\nResolution comparison:")
print(res_metrics.to_string(index=False))

# UMAP plot
import matplotlib.pyplot as plt
sc.pl.umap(adata_scvi, color=['leiden', 'vascular_subtype_prelim', 'sample'],
           ncols=3, show=True)

## Cell 5: Script 03 — EC Subtype Annotation

In [ ]:
%run ../src/03_annotate_clusters.py

In [ ]:
# Annotation checkpoint
import scanpy as sc
import pandas as pd

adata_ann = sc.read_h5ad("data/adata_annotated.h5ad")
annot = pd.read_csv("data/cluster_annotations.csv")

print("Cell type distribution:")
print(adata_ann.obs['cell_type'].value_counts())
print(f"\nHigh confidence annotations: {(annot['confidence']=='high').sum()}/{len(annot)}")

sc.pl.umap(adata_ann, color=['cell_type', 'annotation_confidence'],
           ncols=2, show=True)

## Cell 6: Script 04 — Raynaud's Signature Scoring

In [ ]:
%run ../src/04_signature_scoring.py

In [ ]:
# Signature scoring checkpoint
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

adata_scored = sc.read_h5ad("data/adata_scored.h5ad")
scores = pd.read_csv("data/signature_scores.csv", index_col=0)

print("Signature scores (top clusters by Raynaud's primary score):")
print(scores.sort_values('raynaud_primary_score', ascending=False).head(8).round(4).to_string())

# Score columns for UMAP
score_cols = [c for c in adata_scored.obs.columns if c.startswith('score_')]
if score_cols:
    sc.pl.umap(adata_scored, color=score_cols, ncols=3,
               vmin=0, vmax='p99', show=True)

# Subtype scores if available
subtype_path = "data/signature_scores_bytype.csv"
import os
if os.path.exists(subtype_path):
    print("\nSignature scores by EC subtype:")
    print(pd.read_csv(subtype_path).round(4).to_string(index=False))

## Cell 7: Script 05 — Differential Expression

In [ ]:
%run ../src/05_differential_expression.py

In [ ]:
# DE checkpoint
import pandas as pd

de = pd.read_csv("data/de_top_genes.csv")
print(f"DE results: {len(de):,} gene-cluster pairs")
print(f"Clusters: {de['cluster'].nunique()}")
print(f"\nTop 10 upregulated genes in cluster 0:")
print(de[(de['cluster']=='0') & (de['direction']=='up')].head(10)['gene'].tolist())

## Cell 8: Script 06 — LINCS L1000 Reversal Scoring
**This step makes many API calls to Enrichr. Expect 15-45 min.**

In [ ]:
%run ../src/06_lincs_repurposing.py

In [ ]:
# LINCS checkpoint
import pandas as pd

candidates = pd.read_csv("data/lincs_candidates.csv")
print(f"LINCS candidates: {len(candidates)}")
print(f"\nTop 15 by reversal score:")
print(candidates.head(15).round(2).to_string(index=False))

## Cell 9: Script 07 — Novelty Assessment + Priority Scoring

In [ ]:
%run ../src/07_novelty_prioritization.py

In [ ]:
# Final results checkpoint
import pandas as pd

priority = pd.read_csv("data/priority_candidates.csv")
patent_watch = pd.read_csv("data/patent_watch.csv")

print(f"Total priority candidates: {len(priority)}")
print(f"Patent watch (NOVEL_ALL, high reversal): {len(patent_watch)}")

print(f"\nNovelty breakdown:")
print(priority['novelty_tier'].value_counts().to_string())

print(f"\nTop 20 priority candidates:")
cols = ['compound','moa','novelty_tier','max_reversal_score','n_clusters','priority_score']
if 'best_cluster_celltype' in priority.columns:
    cols.insert(2,'best_cluster_celltype')
print(priority[cols].head(20).round(2).to_string(index=False))

if not patent_watch.empty:
    print(f"\nPatent watch list:")
    print(patent_watch[cols].to_string(index=False))

## Cell 10: Save outputs to Google Drive

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_OUTPUT = "/content/drive/MyDrive/Ritschel_Research/raynaud_crest_output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy key output files
outputs_to_save = [
    "data/priority_candidates.csv",
    "data/patent_watch.csv",
    "data/signature_scores.csv",
    "data/signature_scores_bytype.csv",
    "data/cluster_annotations.csv",
    "data/resolution_metrics.csv",
    "data/novelty_raw.csv",
    "data/adata_scored.h5ad",
    "data/adata_annotated.h5ad",
]

for f in outputs_to_save:
    if os.path.exists(f):
        dest = os.path.join(DRIVE_OUTPUT, os.path.basename(f))
        shutil.copy2(f, dest)
        print(f"Saved: {os.path.basename(f)}")
    else:
        print(f"Not found (skipping): {f}")

print(f"\nAll outputs saved to {DRIVE_OUTPUT}")

---
## Pipeline complete

Review `data/priority_candidates.csv` for drug candidates.  
Review `data/patent_watch.csv` for NOVEL_ALL high-priority candidates before uploading results to Zenodo.

**Next steps:**
1. Review priority candidates for mechanistic coherence with Raynaud's pathophysiology
2. Assess NOVEL_ALL candidates for provisional patent filing
3. Draft paper following calcinosis paper structure
4. File provisional patent before Zenodo upload